[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohsennasab/python-fundamentals-hh/blob/main/notebooks/09_naip-imagery/09_04_advanced_watershed_mosaic.ipynb)

# Module 9, Optional Advanced Lesson: NAIP Mosaic for a Whole Watershed
## Build a free aerial-imagery workflow from the underlying web services

> **Where this fits:** This is an optional extension to Module 9. Lessons 1 to 3 work on a small teaching AOI with `pystac-client`. This notebook does the full-watershed version the long way: raw STAC HTTP requests, pagination, year-by-year coverage checks, remote window reads, and a clipped mosaic. Work through Lesson 1 first. None of Lessons 1 to 3 depend on this notebook.

### Welcome!

In this lesson, you will retrieve National Agriculture Imagery Program (NAIP) aerial imagery for one or more watersheds. You will search the catalog, check which acquisition year covers the full area, read the required parts of remote imagery tiles, create a clipped mosaic, and export a traceable GeoTIFF.

This is an intermediate lesson. The code uses functions, dictionaries, loops, coordinate transformations, and raster metadata. Each helper is introduced separately so you can follow the data from the catalog request to the final image.

### What You Will Work Through Today

- Understand NAIP, STAC metadata, raster assets, and Cloud Optimized GeoTIFFs
- Select one or more HUC-12 watersheds as the area of interest
- Search the NAIP catalog with a direct HTTP request
- Check coverage by acquisition year before choosing imagery
- Estimate raster size before reading high-resolution data
- Request short-lived anonymous access without exposing the token
- Validate remote tiles before mosaicking them
- Reproject, mosaic, clip, and write a four-band GeoTIFF
- Review natural-color and color-infrared views
- Export a source-tile inventory for traceability

### Module Structure

1. Mental model and engineering use
2. Source, access method, and limitations
3. Area-of-interest preparation
4. Raw STAC search
5. Acquisition coverage QA/QC
6. Output-size planning
7. Remote COG reads and mosaicking
8. Output validation and visualization
9. Export, cautions, and practice

## Learning Objectives and Prerequisites

By the end of the lesson, you will be able to:

- Build and inspect a STAC Item Search request.
- Follow a paginated JSON response.
- Explain the difference between an imagery item and its raster asset.
- Measure whether one NAIP acquisition year covers an entire watershed.
- Read and validate remote Cloud Optimized GeoTIFFs.
- Estimate memory needs from bounds, resolution, and band count.
- Build a projected, clipped NAIP mosaic.
- Verify raster CRS, cell size, bands, valid area, and acquisition metadata.

### Prerequisites

You should be comfortable with:

- Module 3 vector data and coordinate reference systems.
- Module 4 raster metadata and clipping.
- Python functions, lists, dictionaries, loops, and exceptions.

The lesson does not use `earthfetch`, `pystac-client`, the Planetary Computer Python client, or another imagery-download package. It does use general-purpose packages for HTTP requests, geometry, raster IO, tables, and plotting. Rewriting a GeoTIFF reader or a coordinate transformation engine would hide the engineering workflow under file-format details.

## Using an AI Assistant in This Module

An AI assistant can help you trace a request, explain a raster transform, or diagnose a failed QA/QC check. Keep each question tied to the data and the engineering decision.

Useful starting questions:

- *"Trace one NAIP tile from a STAC Item to the final watershed GeoTIFF. Name every validation step."*
- *"Explain why the newest individual tile may not represent the newest complete watershed acquisition."*
- *"My STAC search found imagery, but the coverage table is below 99 percent. What should I inspect before downloading?"*
- *"Explain how output resolution and band count affect array memory and file size."*

Check suggested answers against the printed metadata and the official source documentation. Do not paste a temporary access token into a prompt, message, notebook output, or issue report.

## Part 1: Mental Model

### A Catalog Card and the File It Points To

Think of a STAC Item as a catalog card for one aerial image tile. The card contains:

- Where the tile is located
- When it was acquired
- Its ground sample distance
- Its coordinate reference system
- Links to one or more files

The image file is an **asset** attached to that card. Searching STAC returns metadata first. It does not automatically download every image.

### Why Cloud Optimized GeoTIFFs Matter

A Cloud Optimized GeoTIFF, or COG, is arranged so software can request only the byte ranges needed for an area and display resolution. For this watershed, Rasterio can read the overlapping windows and lower-resolution overviews instead of transferring every full source tile.

### Four Services and Objects

| Term | Role in this lesson |
|---|---|
| STAC collection | The searchable NAIP dataset |
| STAC Item | Metadata for one source imagery tile |
| Image asset | The four-band COG referenced by the Item |
| Access token | A short-lived query string that authorizes reads from the hosted file |

The catalog search and the raster read are separate steps. A successful catalog response does not prove that the asset opens as a valid raster.

## Part 2: Engineering Context and Data Source

NAIP is aerial imagery collected during the agricultural growing season. It is not satellite imagery. The USDA dataset listing states that source resolution has changed over time, including 1 or 2 meter imagery in earlier years and 0.6 or 0.3 meter imagery in newer acquisitions.

Water resources engineers can use NAIP to review:

- Development and roadway patterns
- Visible drainage pathways and channel condition
- Riparian cover and erosion features
- Detention facilities and large hydraulic structures
- Changes that deserve field review or comparison with local GIS

Imagery alone does not provide elevations, drainage connectivity, calibrated roughness, or a design impervious percentage.

### Sources

- [USDA NAIP dataset listing](https://catalog.data.gov/dataset/national-agriculture-imagery-program-naip-imagery)
- [USDA NAIP GeoHub](https://naip-usdaonline.hub.arcgis.com/)
- [Planetary Computer NAIP collection metadata](https://planetarycomputer.microsoft.com/api/stac/v1/collections/naip)
- [STAC Item Search specification](https://api.stacspec.org/v1.0.0/item-search/)
- [Planetary Computer data-access documentation](https://planetarycomputer.microsoft.com/docs/concepts/sas/)

The imagery is produced by the USDA Farm Service Agency. Microsoft Planetary Computer hosts a STAC catalog and COG copy used for this exercise. The STAC API is publicly searchable. The raster assets use short-lived access tokens and remain subject to the host's service terms.

### Today's Engineering Scenario

You are reviewing the Town of South Greeley HUC-12 near Cheyenne, Wyoming. You need one recent, spatially complete aerial-imagery mosaic for watershed screening. The source item IDs, dates, resolution, and access date must remain traceable.

## Part 3: Workspace Setup

The first code cell imports general-purpose tools already used in the course. No specialized STAC or NAIP client is installed.

The imports are grouped by job. This makes it easier to identify which tool handles HTTP, vector geometry, raster data, tables, or plotting.

In [ ]:
# Standard-library tools
import datetime
import math
import time
from pathlib import Path

# HTTP and tabular data
import requests
import numpy as np
import pandas as pd

# Vector geometry
import geopandas as gpd
from shapely.geometry import mapping, shape
from shapely.ops import unary_union

# Raster processing
import rasterio
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from rasterio.plot import plotting_extent
from rasterio.transform import Affine, array_bounds, from_origin
from rasterio.warp import reproject, transform_bounds
from rasterio.windows import Window, WindowError, from_bounds

# Plotting
import matplotlib.pyplot as plt
from matplotlib import font_manager

# Every figure in this lesson uses Arial. Colab runs on Linux, which does not
# ship Arial, so the list falls back to Liberation Sans: a font drawn with the
# same letter widths and shapes, made as a drop-in replacement. Naming a font
# that does exist is also what keeps matplotlib from printing a warning for
# every label it draws.
FONT_PREFERENCE = ['Arial', 'Liberation Sans', 'Helvetica', 'DejaVu Sans']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = FONT_PREFERENCE

installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
active_font = next((name for name in FONT_PREFERENCE if name in installed_fonts), None)
print(f"Figure font: {active_font or 'matplotlib default'}")

print("Libraries imported. Ready to prepare the area of interest.")

### Set the Lesson Controls

The settings below are kept together so the workflow can be adapted without searching through every code cell.

- `TARGET_HUCS` can contain one or more HUC-12 identifiers.
- `REQUESTED_YEAR = None` tells the notebook to choose the newest year that passes the coverage threshold.
- `OUTPUT_RESOLUTION_M = 2.0` creates a manageable watershed-scale teaching file.
- `MINIMUM_COVERAGE_PCT` is the minimum footprint coverage required before a year can be selected.

The output resolution is not the source resolution. The source ground sample distance is read from each STAC Item and retained in the tile inventory.

In [ ]:
# One or more HUC-12 identifiers from the course watershed file
TARGET_HUCS = ['101900090108']

# Use None for the newest complete acquisition, or enter a year such as 2019
REQUESTED_YEAR = None

# A coarser output keeps a watershed mosaic practical for a course exercise
OUTPUT_RESOLUTION_M = 2.0

# Require almost complete footprint coverage before selecting an acquisition
MINIMUM_COVERAGE_PCT = 99.0

# Stop before building an unexpectedly large in-memory raster
MAXIMUM_UNCOMPRESSED_MB = 600

# Public endpoints used by the raw HTTP workflow
STAC_SEARCH_URL = (
    'https://planetarycomputer.microsoft.com/api/stac/v1/search'
)
NAIP_TOKEN_URL = (
    'https://planetarycomputer.microsoft.com/api/sas/v1/token/naip'
)

print(f"Target HUC-12 count: {len(TARGET_HUCS)}")
print(f"Requested year: {REQUESTED_YEAR or 'newest complete acquisition'}")
print(f"Output resolution: {OUTPUT_RESOLUTION_M:g} meters")

### Locate the Watershed File

The notebook first looks for the public course ZIP in common local paths. If it is not present, a Colab file picker appears.

The expected file is `NHD__Watershed_Boundaries_HUC_12_Selected.zip`. Reading the shapefile directly from the ZIP keeps its component files together.

In [ ]:
# Search local paths before asking for an upload
watershed_filename = 'NHD__Watershed_Boundaries_HUC_12_Selected.zip'
watershed_candidates = [
    Path(watershed_filename),
    Path('data') / watershed_filename,
    Path('notebooks/09_naip-imagery/data') / watershed_filename,
]

watershed_zip = next(
    (path for path in watershed_candidates if path.exists()),
    None,
)

if watershed_zip is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            f"Could not find {watershed_filename}. Place it in the "
            "current folder or the module data folder."
        ) from exc

    print(f"Please upload {watershed_filename}.")
    uploaded = files.upload()
    if watershed_filename not in uploaded:
        raise FileNotFoundError(
            f"The selected upload did not include {watershed_filename}."
        )
    watershed_zip = Path(watershed_filename)

print(f"Watershed source: {watershed_zip}")

### Select and Check the Area of Interest

The next cell selects every identifier in `TARGET_HUCS`, dissolves them into one analysis geometry, and checks that the geometry and CRS are usable.

Three coordinate systems have different jobs:

- The source watershed CRS is retained only for the initial read.
- EPSG:4326 is used for the STAC search because GeoJSON coordinates are longitude and latitude.
- A local UTM CRS is used for area calculations and raster output because its units are meters.

The printed count and area are the first protection against selecting the wrong watershed.

In [ ]:
# Read all course watersheds directly from the ZIP archive
watersheds = gpd.read_file(f'zip://{watershed_zip}')

if watersheds.crs is None:
    raise RuntimeError(
        "The watershed file has no CRS. Confirm that the ZIP includes its .prj file."
    )
if 'HUC12' not in watersheds.columns:
    raise RuntimeError("The watershed file does not contain an HUC12 field.")

# Convert identifiers to text before matching so leading zeros are preserved
selected_watersheds = watersheds[
    watersheds['HUC12'].astype(str).isin(TARGET_HUCS)
].copy()

if len(selected_watersheds) != len(set(TARGET_HUCS)):
    found_hucs = set(selected_watersheds['HUC12'].astype(str))
    missing_hucs = sorted(set(TARGET_HUCS) - found_hucs)
    raise RuntimeError(f"These HUC-12 identifiers were not found: {missing_hucs}")
if selected_watersheds.geometry.is_empty.any():
    raise RuntimeError("At least one selected watershed has an empty geometry.")
if not selected_watersheds.geometry.is_valid.all():
    raise RuntimeError(
        "At least one selected watershed geometry is invalid. Repair it before continuing."
    )

# Dissolve several watersheds into one search and clipping geometry
aoi = selected_watersheds.dissolve().reset_index(drop=True)
aoi_wgs84 = aoi.to_crs('EPSG:4326')
output_crs = aoi_wgs84.estimate_utm_crs()
if output_crs is None:
    raise RuntimeError("A local UTM CRS could not be estimated for the area of interest.")
aoi_projected = aoi_wgs84.to_crs(output_crs)

aoi_area_m2 = aoi_projected.geometry.iloc[0].area
aoi_area_acres = aoi_area_m2 / 4046.8564224

print(f"Selected watersheds: {len(selected_watersheds)}")
print(f"HUC-12 identifiers: {', '.join(TARGET_HUCS)}")
print(f"Search CRS: {aoi_wgs84.crs}")
print(f"Output and area CRS: {output_crs}")
print(f"Dissolved area: {aoi_area_acres:,.0f} acres")

The default selection should report one watershed, UTM Zone 13N, and an area of about 12,000 acres.

Stop here if the HUC count, identifiers, CRS, or area are unexpected. A correct download built for the wrong polygon is still the wrong engineering dataset.

## Part 4: Search the NAIP Catalog with Raw HTTP Requests

### Helper 1: Request and Validate JSON

The first helper sends an HTTP request and accepts the result only when:

- The server returns a successful status code.
- The response can be decoded as JSON.
- Repeated temporary failures have been exhausted.

Retries use short waits. The final error reports the endpoint and failure type without printing any future access token.

In [ ]:
# Reuse one HTTP session so repeated requests can share connections
http_session = requests.Session()
http_session.headers.update({
    'User-Agent': 'python-fundamentals-hh-naip-lesson/1.0'
})


def request_json(method, url, json_body=None, attempts=5, timeout=60):
    '''Return validated JSON from an HTTP endpoint, waiting out rate limits.

    A free public service answers a burst of requests with HTTP 429, "too many
    requests". That is not a failure to report to the user; it is an
    instruction to slow down. Each retry waits longer than the last, and when
    the server sends a Retry-After header we wait as long as it asks.

    Colab sends every notebook's requests through a small set of shared
    addresses, so a busy class can reach an anonymous rate limit together
    without any one person doing anything wrong.
    '''
    last_error = None

    for attempt_number in range(1, attempts + 1):
        rate_limited = False
        retry_after = None

        try:
            response = http_session.request(
                method=method,
                url=url,
                json=json_body,
                timeout=timeout,
            )
            # Read these before raise_for_status() turns the response into an error
            rate_limited = response.status_code == 429
            retry_after = response.headers.get('Retry-After')
            response.raise_for_status()

            try:
                return response.json()
            except ValueError as exc:
                raise RuntimeError(
                    "The server response was not valid JSON."
                ) from exc

        except (requests.RequestException, RuntimeError) as exc:
            last_error = exc
            if attempt_number == attempts:
                break

            # A rate limit deserves a longer pause than an ordinary hiccup
            if rate_limited:
                wait_seconds = 2 ** attempt_number
                if retry_after and retry_after.isdigit():
                    wait_seconds = max(wait_seconds, int(retry_after))
                reason = 'was rate limited (HTTP 429)'
            else:
                wait_seconds = 2 ** (attempt_number - 1)
                reason = 'failed'

            print(
                f"  Request {reason}. Waiting {wait_seconds} second(s), "
                f"then attempt {attempt_number + 1} of {attempts}."
            )
            time.sleep(wait_seconds)

    if last_error is not None and '429' in str(last_error):
        raise RuntimeError(
            f"The service is still rate limiting after {attempts} attempts:\n{url}\n"
            "This is not a problem with your code or your area of interest. "
            "Wait a minute or two, then rerun this cell."
        ) from last_error

    raise RuntimeError(
        f"Request failed after {attempts} attempts: {url}\n"
        f"Last error: {last_error}"
    )


print("HTTP JSON helper is ready.")

### Helper 2: Follow STAC Search Pages

A STAC Item Search returns a GeoJSON FeatureCollection. Large searches can be split across pages. The next helper:

1. Sends the area geometry and collection name to `/search`.
2. Confirms that the response is a FeatureCollection.
3. stores each returned Item by its unique ID.
4. Follows the server-provided `next` request body.
5. Stops when there is no next page.

The server controls pagination. Reconstructing page tokens by hand would make the code dependent on an implementation detail.

In [ ]:
def search_stac_items(search_geometry, collection='naip', max_pages=20):
    '''Search a STAC collection and follow server-provided POST pages.'''
    request_body = {
        'collections': [collection],
        'intersects': mapping(search_geometry),
        'limit': 100,
    }
    items_by_id = {}

    for page_number in range(1, max_pages + 1):
        page = request_json(
            'POST',
            STAC_SEARCH_URL,
            json_body=request_body,
        )

        # STAC Item Search should return a GeoJSON FeatureCollection
        if page.get('type') != 'FeatureCollection':
            raise RuntimeError(
                "The STAC response was not a GeoJSON FeatureCollection."
            )
        features = page.get('features')
        if not isinstance(features, list):
            raise RuntimeError("The STAC response has no feature list.")

        # Item IDs prevent accidental duplicates across pages
        for item in features:
            item_id = item.get('id')
            if not item_id:
                raise RuntimeError("A STAC Item is missing its ID.")
            items_by_id[item_id] = item

        print(
            f"STAC page {page_number}: {len(features)} item(s), "
            f"{len(items_by_id)} unique item(s) so far"
        )

        next_link = next(
            (
                link for link in page.get('links', [])
                if link.get('rel') == 'next'
            ),
            None,
        )
        if next_link is None:
            return list(items_by_id.values())

        # Planetary Computer provides the next page as another POST body
        if next_link.get('method', 'GET').upper() != 'POST':
            raise RuntimeError(
                "The STAC server returned an unsupported pagination method."
            )
        request_body = next_link.get('body')
        if not isinstance(request_body, dict):
            raise RuntimeError(
                "The STAC next link did not contain a request body."
            )

    raise RuntimeError(
        f"The STAC search exceeded the {max_pages}-page safety limit."
    )


print("STAC pagination helper is ready.")

### Run the Spatial Search

The search uses the dissolved watershed geometry, not only its rectangular bounds. This reduces tiles that touch the bounding box but do not actually intersect the watershed.

The result is metadata. No imagery has been transferred yet.

In [ ]:
# Search every NAIP Item whose footprint intersects the watershed geometry
naip_items = search_stac_items(
    aoi_wgs84.geometry.iloc[0],
    collection='naip',
)

if not naip_items:
    raise RuntimeError(
        "No NAIP Items intersect the area of interest. "
        "Confirm that the AOI is within NAIP coverage."
    )

print(f"\nTotal unique NAIP Items found: {len(naip_items)}")

The default watershed should return several dozen Items from multiple acquisition years. A larger count does not mean more complete coverage. It may simply mean the area has been flown several times.

**Question to ask your AI assistant:** *"Explain the difference between a STAC search returning 35 Items and the watershed being covered by 35 images at one time. Which Item fields are needed to separate those ideas?"*

## Part 5: Build an Acquisition Inventory

### Convert Item Metadata to a GeoDataFrame

The next helper extracts only the fields needed for engineering review:

- Item ID
- Acquisition date and NAIP year
- Ground sample distance
- Source EPSG code
- State
- Number of image bands
- Tile footprint

It also requires a four-band image asset. A thumbnail or rendered preview is not accepted as analysis data.

In [ ]:
def build_naip_inventory(items):
    '''Convert NAIP STAC Items into a reviewable spatial inventory.'''
    records = []

    for item in items:
        properties = item.get('properties', {})
        image_asset = item.get('assets', {}).get('image')
        if image_asset is None:
            raise RuntimeError(
                f"STAC Item {item.get('id')} has no image asset."
            )

        item_geometry = item.get('geometry')
        if item_geometry is None:
            raise RuntimeError(
                f"STAC Item {item.get('id')} has no footprint geometry."
            )

        band_metadata = image_asset.get('eo:bands', [])
        records.append({
            'item_id': item['id'],
            'acquisition_datetime': pd.to_datetime(
                properties.get('datetime'),
                utc=True,
            ),
            'naip_year': int(properties['naip:year']),
            'ground_sample_distance_m': float(properties['gsd']),
            'source_epsg': int(properties['proj:epsg']),
            'state': properties.get('naip:state', '').upper(),
            'band_count': len(band_metadata),
            'asset_type': image_asset.get('type', ''),
            'geometry': shape(item_geometry),
        })

    inventory = gpd.GeoDataFrame(
        records,
        geometry='geometry',
        crs='EPSG:4326',
    )

    if inventory['acquisition_datetime'].isna().any():
        raise RuntimeError("At least one NAIP Item has no acquisition date.")
    if (inventory['band_count'] < 4).any():
        raise RuntimeError(
            "At least one selected NAIP asset has fewer than four bands."
        )

    return inventory


naip_inventory = build_naip_inventory(naip_items)

inventory_summary = (
    naip_inventory
    .groupby('naip_year')
    .agg(
        tile_count=('item_id', 'count'),
        first_acquisition=('acquisition_datetime', 'min'),
        last_acquisition=('acquisition_datetime', 'max'),
        minimum_gsd_m=('ground_sample_distance_m', 'min'),
        maximum_gsd_m=('ground_sample_distance_m', 'max'),
    )
    .reset_index()
    .sort_values('naip_year', ascending=False)
)

inventory_summary

The inventory table shows how many source tiles intersect the watershed in each year. The first and last acquisition dates can differ because a mosaic may cross flight dates.

Ground sample distance is a source property. It should not be replaced by the output cell size selected later.

### Measure Watershed Coverage by Year

The newest year in the table is not automatically suitable. One recent tile might touch a corner of an area while an earlier acquisition covers it completely.

For each year, the next cell:

1. Reprojects the tile footprints to the local UTM CRS.
2. Unions overlapping footprints so overlap is not counted twice.
3. intersects the footprint union with the watershed.
4. Divides covered area by watershed area.

This calculation uses footprints only. The final raster receives a second valid-pixel coverage check after it is written.

In [ ]:
# Use a projected CRS so footprint and watershed areas are in square meters
inventory_projected = naip_inventory.to_crs(output_crs)
aoi_geometry_projected = aoi_projected.geometry.iloc[0]

coverage_records = []
for year, year_tiles in inventory_projected.groupby('naip_year'):
    # Union removes tile overlap before area is measured
    footprint_union = unary_union(year_tiles.geometry.tolist())
    covered_geometry = footprint_union.intersection(aoi_geometry_projected)
    coverage_pct = 100 * covered_geometry.area / aoi_area_m2

    coverage_records.append({
        'naip_year': int(year),
        'tile_count': len(year_tiles),
        'coverage_pct': min(coverage_pct, 100.0),
        'minimum_gsd_m': year_tiles[
            'ground_sample_distance_m'
        ].min(),
        'maximum_gsd_m': year_tiles[
            'ground_sample_distance_m'
        ].max(),
    })

coverage_by_year = (
    pd.DataFrame(coverage_records)
    .sort_values('naip_year', ascending=False)
    .reset_index(drop=True)
)
coverage_by_year['passes_coverage_check'] = (
    coverage_by_year['coverage_pct'] >= MINIMUM_COVERAGE_PCT
)

coverage_by_year.round({
    'coverage_pct': 2,
    'minimum_gsd_m': 2,
    'maximum_gsd_m': 2,
})

For the default watershed, complete years should be near 100 percent coverage. Any year below 99 percent is rejected by the lesson even if it is the newest date returned by the catalog.

Footprint coverage does not prove that every raster pixel is readable. It is the first selection check.

**Question to ask your AI assistant:** *"Why do we union tile footprints before calculating coverage? Give an example showing how summing each tile's intersection area could produce more than 100 percent."*

### Select One Complete Acquisition Year

If `REQUESTED_YEAR` is `None`, the code selects the newest year meeting the threshold. If a year was entered, that year must exist and pass the same check.

Keeping one year avoids a patchwork of older and newer imagery. Adjacent tiles can still have different dates within the selected acquisition year, so the tile inventory remains important.

In [ ]:
eligible_years = coverage_by_year.loc[
    coverage_by_year['passes_coverage_check'],
    'naip_year',
].tolist()

if not eligible_years:
    raise RuntimeError(
        f"No acquisition year covers at least "
        f"{MINIMUM_COVERAGE_PCT:.1f}% of the AOI."
    )

if REQUESTED_YEAR is None:
    selected_year = max(eligible_years)
else:
    selected_year = int(REQUESTED_YEAR)
    if selected_year not in coverage_by_year['naip_year'].tolist():
        raise RuntimeError(
            f"Requested year {selected_year} was not found in the STAC search."
        )
    if selected_year not in eligible_years:
        requested_coverage = coverage_by_year.loc[
            coverage_by_year['naip_year'] == selected_year,
            'coverage_pct',
        ].iloc[0]
        raise RuntimeError(
            f"Requested year {selected_year} covers only "
            f"{requested_coverage:.2f}% of the AOI."
        )

selected_inventory = (
    naip_inventory[naip_inventory['naip_year'] == selected_year]
    .sort_values('acquisition_datetime', ascending=False)
    .copy()
)
selected_item_ids = set(selected_inventory['item_id'])
selected_items = [
    item for item in naip_items
    if item['id'] in selected_item_ids
]

selected_coverage_pct = coverage_by_year.loc[
    coverage_by_year['naip_year'] == selected_year,
    'coverage_pct',
].iloc[0]

print(f"Selected NAIP year: {selected_year}")
print(f"Selected source tiles: {len(selected_items)}")
print(f"Footprint coverage: {selected_coverage_pct:.2f}%")
print(
    "Acquisition dates: "
    f"{selected_inventory['acquisition_datetime'].min().date()} to "
    f"{selected_inventory['acquisition_datetime'].max().date()}"
)

Read all four lines before continuing. The footprint coverage must meet the threshold, every tile must have the selected year, and the date range should be recorded for later interpretation.

The default watershed should select 2022 rather than choosing an isolated newer tile from a nearby acquisition.

### Map the Selected Tile Footprints

The footprint map is a spatial QA/QC check before any large raster read. Tile boundaries should collectively cover the watershed polygon.

The numbers correspond to rows in the selected tile inventory. They are labels for this lesson, not permanent NAIP identifiers.

In [ ]:
selected_footprints = selected_inventory.to_crs(output_crs).copy()
selected_footprints['tile_number'] = range(
    1, len(selected_footprints) + 1
)

fig, ax = plt.subplots(figsize=(9, 7))

# Draw imagery footprints first so the watershed remains visible on top
selected_footprints.boundary.plot(
    ax=ax,
    color='#4C78A8',
    linewidth=1.2,
)
aoi_projected.boundary.plot(
    ax=ax,
    color='#C43C39',
    linewidth=2.2,
)

# Label footprint centroids for comparison with the inventory table
for _, row in selected_footprints.iterrows():
    centroid = row.geometry.centroid
    ax.text(
        centroid.x,
        centroid.y,
        str(row['tile_number']),
        ha='center',
        va='center',
        fontsize=9,
        color='#1F1F1F',
    )

ax.set_title(
    f"NAIP {selected_year} Source Tile Footprints",
    fontweight='normal',
)
ax.set_xlabel(f"Easting, meters ({output_crs.to_string()})")
ax.set_ylabel(f"Northing, meters ({output_crs.to_string()})")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

selected_footprints[[
    'tile_number',
    'item_id',
    'acquisition_datetime',
    'ground_sample_distance_m',
    'source_epsg',
]]

The red watershed outline should remain inside the combined blue footprints. Overlap between neighboring source tiles is normal.

Visual coverage is useful, but the numeric footprint percentage is the selection criterion. The raster mask check later is the final coverage criterion.

## Part 6: Estimate the Output Before Downloading

High-resolution imagery can exhaust memory quickly. Array size depends on:

`width x height x band count x bytes per value`

The next cell estimates the rectangular output grid. The rectangle includes pixels outside the watershed until the polygon mask is applied, so it is intentionally conservative.

This course uses four one-byte bands. A native 0.6 meter watershed mosaic would contain about eleven times as many pixels as a 2 meter mosaic.

In [ ]:
minx, miny, maxx, maxy = aoi_projected.total_bounds
estimated_width = math.ceil(
    (maxx - minx) / OUTPUT_RESOLUTION_M
)
estimated_height = math.ceil(
    (maxy - miny) / OUTPUT_RESOLUTION_M
)
output_band_count = 4
bytes_per_value = 1

estimated_uncompressed_mb = (
    estimated_width
    * estimated_height
    * output_band_count
    * bytes_per_value
    / 1024 ** 2
)

source_gsd_min = selected_inventory[
    'ground_sample_distance_m'
].min()
source_gsd_max = selected_inventory[
    'ground_sample_distance_m'
].max()

print(
    f"Selected source ground sample distance: "
    f"{source_gsd_min:g} to {source_gsd_max:g} meters"
)
print(
    f"Estimated output grid: "
    f"{estimated_width:,} columns x {estimated_height:,} rows"
)
print(f"Output bands: {output_band_count} (red, green, blue, NIR)")
print(
    f"Estimated uncompressed array: "
    f"{estimated_uncompressed_mb:,.1f} MB"
)

if OUTPUT_RESOLUTION_M < source_gsd_min:
    print(
        "WARNING: The output cell size is finer than the source GSD. "
        "This adds pixels but does not add source detail."
    )
if estimated_uncompressed_mb > MAXIMUM_UNCOMPRESSED_MB:
    raise RuntimeError(
        f"The estimated array exceeds {MAXIMUM_UNCOMPRESSED_MB} MB. "
        "Use a coarser output resolution, a smaller AOI, or a tiled workflow."
    )

print("Output-size QA/QC passed.")

The estimate is memory for one array, not a promise about final compressed file size or total peak memory. Mosaicking, masks, previews, and library buffers require additional memory.

For design-scale review at native resolution, divide large watersheds into processing tiles or use a smaller project AOI. Do not raise the memory limit until you understand the expected dimensions.

**Question to ask your AI assistant:** *"Recalculate the estimated array size at 1 meter and 0.6 meter resolution. Explain why reducing cell size by half creates about four times as many pixels."*

## Part 7: Request Temporary Raster Access

### Helper 3: Obtain and Apply a Short-Lived Token

The STAC search is public, but the hosted NAIP raster links need a short-lived Shared Access Signature.

The helper requests one token for the NAIP collection and appends it to each image URL only in memory. It returns the expiration time for troubleshooting but never prints or exports the token.

Treat the token like a temporary credential:

- Do not print it.
- Do not save signed URLs in a CSV.
- Do not commit it.
- Request a new token if it expires during a long session.

In [ ]:
def request_naip_access():
    '''Return a temporary NAIP access token and its expiration time.'''
    token_response = request_json(
        'GET',
        NAIP_TOKEN_URL,
        attempts=3,
        timeout=30,
    )

    token = token_response.get('token')
    expiration = token_response.get('msft:expiry')
    if not token or not expiration:
        raise RuntimeError(
            "The access response did not include a token and expiration."
        )

    return token, expiration


def add_access_token(asset_href, token):
    '''Append an access token to an asset URL without logging it.'''
    separator = '&' if '?' in asset_href else '?'
    return f"{asset_href}{separator}{token}"


print("Temporary-access helpers are ready. No token has been requested yet.")

The helpers define the access step but do not run it yet. The token is requested immediately before the raster reads so as much of its short lifetime as possible remains available.

**Question to ask your AI assistant:** *"Explain why the notebook stores unsigned Item IDs in its inventory but keeps signed raster URLs only inside the download function."*

### Helper 4: Validate Each Remote Raster

An asset URL is accepted only after Rasterio opens it as a georeferenced four-band, unsigned-byte raster.

The function also compares the opened raster with key STAC metadata:

- Raster band count
- Raster CRS and STAC EPSG
- Raster cell size and STAC ground sample distance

A mismatch raises an error before any mosaic is built.

In [ ]:
def validate_naip_source(source, item):
    '''Validate an opened NAIP source raster against its STAC Item.'''
    item_id = item['id']
    properties = item['properties']
    expected_epsg = int(properties['proj:epsg'])
    expected_gsd = float(properties['gsd'])

    problems = []
    if source.driver != 'GTiff':
        problems.append(f"expected GTiff, found {source.driver}")
    if source.count < 4:
        problems.append(f"expected at least 4 bands, found {source.count}")
    if any(dtype != 'uint8' for dtype in source.dtypes[:4]):
        problems.append(
            f"expected uint8 bands, found {source.dtypes[:4]}"
        )
    if source.crs is None:
        problems.append("raster has no CRS")
    elif source.crs.to_epsg() != expected_epsg:
        problems.append(
            f"STAC EPSG is {expected_epsg}, raster EPSG is "
            f"{source.crs.to_epsg()}"
        )
    elif not source.crs.is_projected:
        problems.append("source CRS is not projected")
    if not np.allclose(
        np.abs(source.res),
        (expected_gsd, expected_gsd),
        atol=0.01,
    ):
        problems.append(
            f"STAC GSD is {expected_gsd}, raster resolution is "
            f"{source.res}"
        )
    if source.width < 1 or source.height < 1:
        problems.append("raster has no cells")

    if problems:
        raise RuntimeError(
            f"NAIP source validation failed for {item_id}:\n- "
            + "\n- ".join(problems)
        )

    return {
        'item_id': item_id,
        'raster_epsg': source.crs.to_epsg(),
        'raster_width': source.width,
        'raster_height': source.height,
        'raster_resolution_x_m': abs(source.res[0]),
        'raster_resolution_y_m': abs(source.res[1]),
        'raster_band_count': source.count,
        'raster_dtype': source.dtypes[0],
        'overview_levels': ','.join(
            str(level) for level in source.overviews(1)
        ),
    }


print("Remote-raster validation helper is ready.")

The validator checks the source file itself, not only the catalog record. This catches expired access, non-raster responses, missing georeferencing, unexpected band layouts, and metadata mismatches.

### Helper 5: Write a Traceable GeoTIFF

The output writer saves:

- Four unsigned-byte bands in red, green, blue, and near-infrared order
- The projected transform and CRS
- An internal valid-data mask
- Tiling, lossless compression, and overviews
- Source collection, year, Item IDs, acquisition dates, output resolution, and access date

The access token is deliberately absent from the metadata.

In [ ]:
def write_naip_geotiff(
    output_path,
    mosaic,
    valid_mask,
    transform,
    crs,
    selected_items,
    output_resolution_m,
):
    '''Write the clipped four-band mosaic and traceability metadata.'''
    acquisition_dates = sorted({
        item['properties']['datetime'][:10]
        for item in selected_items
    })
    item_ids = sorted(item['id'] for item in selected_items)

    profile = {
        'driver': 'GTiff',
        'height': mosaic.shape[1],
        'width': mosaic.shape[2],
        'count': 4,
        'dtype': 'uint8',
        'crs': crs,
        'transform': transform,
        'compress': 'deflate',
        'predictor': 2,
        'tiled': True,
        'blockxsize': 512,
        'blockysize': 512,
        'interleave': 'pixel',
        'BIGTIFF': 'IF_SAFER',
    }

    # Keep the polygon and source-data mask inside the GeoTIFF
    with rasterio.Env(GDAL_TIFF_INTERNAL_MASK=True):
        with rasterio.open(output_path, 'w', **profile) as destination:
            destination.write(mosaic)
            destination.write_mask(
                valid_mask.astype('uint8') * 255
            )

            for band_number, band_name in enumerate(
                ['Red', 'Green', 'Blue', 'Near infrared'],
                start=1,
            ):
                destination.set_band_description(
                    band_number,
                    band_name,
                )

            destination.update_tags(
                source_program='USDA National Agriculture Imagery Program',
                source_host='Microsoft Planetary Computer',
                stac_collection='naip',
                naip_year=str(selected_year),
                source_item_ids='|'.join(item_ids),
                acquisition_dates='|'.join(acquisition_dates),
                access_date=datetime.date.today().isoformat(),
                output_resolution_m=str(output_resolution_m),
                processing=(
                    'Remote COG window read, bilinear reprojection, '
                    'newer-date overlap priority, AOI clip'
                ),
                source_reference=(
                    'https://catalog.data.gov/dataset/'
                    'national-agriculture-imagery-program-naip-imagery'
                ),
            )

            # Overviews make later display faster at small map scales
            overview_factors = [
                factor for factor in [2, 4, 8, 16]
                if min(mosaic.shape[1:]) // factor >= 128
            ]
            if overview_factors:
                destination.build_overviews(
                    overview_factors,
                    Resampling.average,
                )
                destination.update_tags(
                    ns='rio_overview',
                    resampling='average',
                )


print("GeoTIFF writer is ready.")

### Helper 6: Read, Reproject, Mosaic, and Clip

This helper performs the main transfer. Its sequence is intentionally explicit:

1. Request one temporary token.
2. Open each selected COG and validate it.
3. Calculate the source window needed for the output grid.
4. Read that window at an overview-sized resolution and reproject it.
5. Apply the watershed mask.
6. Check valid raster coverage.
7. Write the final GeoTIFF.

Source Items are processed from older to newer acquisition dates within the selected year. A newer tile therefore replaces an older tile only where their buffered edges overlap. Pixel values are not averaged across seams.

In [ ]:
def create_naip_subset(
    items,
    aoi_projected,
    output_path,
    output_resolution_m,
    minimum_coverage_pct,
):
    '''Create and validate a clipped NAIP mosaic from selected STAC Items.'''
    access_token, token_expiration = request_naip_access()
    print(f"Temporary raster access expires: {token_expiration}")

    # Process older dates first so newer imagery wins in overlapping buffers
    ordered_items = sorted(
        items,
        key=lambda item: item['properties']['datetime'],
        reverse=False,
    )

    # Align the output rectangle to whole cells at the requested resolution
    minx, miny, maxx, maxy = aoi_projected.total_bounds
    left = math.floor(minx / output_resolution_m) * output_resolution_m
    bottom = math.floor(miny / output_resolution_m) * output_resolution_m
    right = math.ceil(maxx / output_resolution_m) * output_resolution_m
    top = math.ceil(maxy / output_resolution_m) * output_resolution_m
    output_width = round((right - left) / output_resolution_m)
    output_height = round((top - bottom) / output_resolution_m)
    output_transform = from_origin(
        left,
        top,
        output_resolution_m,
        output_resolution_m,
    )
    output_bounds = array_bounds(
        output_height,
        output_width,
        output_transform,
    )
    output_geometry = aoi_projected.geometry.iloc[0]
    source_qa_records = []
    mosaic = np.zeros(
        (4, output_height, output_width),
        dtype='uint8',
    )

    try:
        with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
            for item in ordered_items:
                unsigned_href = item['assets']['image']['href']
                signed_href = add_access_token(
                    unsigned_href,
                    access_token,
                )

                # Keep signed URLs inside this function and out of logs
                try:
                    source_context = rasterio.open(signed_href)
                except Exception:
                    raise RuntimeError(
                        f"Could not open NAIP raster {item['id']}. "
                        "The service may be unavailable or the token "
                        "may have expired."
                    ) from None

                with source_context as source:
                    source_record = validate_naip_source(source, item)

                    # Limit this read to the part of the AOI covered by
                    # the current STAC Item footprint.
                    item_footprint_output = gpd.GeoSeries(
                        [shape(item['geometry'])],
                        crs='EPSG:4326',
                    ).to_crs(aoi_projected.crs).iloc[0]
                    needed_geometry = (
                        item_footprint_output
                        .intersection(output_geometry)
                    )
                    if needed_geometry.is_empty:
                        continue

                    # Transform the tight intersection bounds to source CRS
                    source_request_bounds = transform_bounds(
                        aoi_projected.crs,
                        source.crs,
                        *needed_geometry.bounds,
                        densify_pts=21,
                    )

                    try:
                        source_window = from_bounds(
                            *source_request_bounds,
                            transform=source.transform,
                        )

                        # Add two source pixels for the bilinear kernel
                        source_window = Window(
                            math.floor(source_window.col_off) - 2,
                            math.floor(source_window.row_off) - 2,
                            math.ceil(source_window.width) + 4,
                            math.ceil(source_window.height) + 4,
                        )
                        source_window = source_window.intersection(
                            Window(
                                0,
                                0,
                                source.width,
                                source.height,
                            )
                        )
                    except WindowError:
                        continue

                    if source_window.width <= 0 or source_window.height <= 0:
                        continue

                    # NAIP source CRSs and the output UTM CRS use meters.
                    # Read near the final grid scale so COG overviews can
                    # satisfy the request without native-resolution transfer.
                    source_cell_size = max(
                        abs(source.res[0]),
                        abs(source.res[1]),
                    )
                    decimation_factor = max(
                        1.0,
                        output_resolution_m / source_cell_size,
                    )
                    read_width = max(
                        1,
                        math.ceil(
                            source_window.width
                            / decimation_factor
                        ),
                    )
                    read_height = max(
                        1,
                        math.ceil(
                            source_window.height
                            / decimation_factor
                        ),
                    )

                    source_data = source.read(
                        [1, 2, 3, 4],
                        window=source_window,
                        out_shape=(4, read_height, read_width),
                        resampling=Resampling.bilinear,
                    )
                    reduced_transform = (
                        source.window_transform(source_window)
                        * Affine.scale(
                            source_window.width / read_width,
                            source_window.height / read_height,
                        )
                    )

                    # Reproject each band into the common output grid.
                    # init_dest_nodata=False preserves pixels written by
                    # earlier tiles outside this tile's footprint.
                    for band_index in range(4):
                        reproject(
                            source=source_data[band_index],
                            destination=mosaic[band_index],
                            src_transform=reduced_transform,
                            src_crs=source.crs,
                            dst_transform=output_transform,
                            dst_crs=aoi_projected.crs,
                            src_nodata=None,
                            dst_nodata=0,
                            resampling=Resampling.bilinear,
                            init_dest_nodata=False,
                            num_threads=2,
                        )

                    source_record.update({
                        'requested_window_width': round(
                            source_window.width
                        ),
                        'requested_window_height': round(
                            source_window.height
                        ),
                        'transferred_array_width': read_width,
                        'transferred_array_height': read_height,
                    })
                    source_qa_records.append(source_record)
                    print(
                        f"Read {item['id']}: "
                        f"{round(source_window.width):,} x "
                        f"{round(source_window.height):,} source cells "
                        f"to {read_width:,} x {read_height:,}"
                    )
    finally:
        # Remove the token reference as soon as remote reads are complete
        access_token = None

    if mosaic.shape[0] != 4:
        raise RuntimeError(
            f"The mosaic has {mosaic.shape[0]} bands instead of 4."
        )

    # Build a center-based mask for the dissolved watershed polygon
    inside_aoi = geometry_mask(
        [mapping(output_geometry)],
        out_shape=mosaic.shape[1:],
        transform=output_transform,
        invert=True,
    )

    # All-zero RGBN pixels are treated as missing for the gap check
    source_data_present = np.any(mosaic != 0, axis=0)
    valid_mask = inside_aoi & source_data_present
    raster_coverage_pct = (
        100 * valid_mask.sum() / inside_aoi.sum()
    )

    if raster_coverage_pct < minimum_coverage_pct:
        raise RuntimeError(
            f"The merged raster covers only {raster_coverage_pct:.2f}% "
            "of AOI cell centers."
        )

    # Set cells outside the AOI or inside a data gap to the working fill value
    mosaic[:, ~valid_mask] = 0

    write_naip_geotiff(
        output_path=output_path,
        mosaic=mosaic,
        valid_mask=valid_mask,
        transform=output_transform,
        crs=aoi_projected.crs,
        selected_items=ordered_items,
        output_resolution_m=output_resolution_m,
    )

    return pd.DataFrame(source_qa_records), raster_coverage_pct


print("NAIP subset helper is ready.")

The long workflow is now divided into separate jobs: HTTP validation, STAC pagination, metadata inventory, temporary access, source-raster validation, output writing, and orchestration.

**Question to ask your AI assistant:** *"Trace the failure path when a STAC Item exists but its signed asset opens as an HTML error instead of a GeoTIFF. Which function stops the workflow, and why does that happen before mosaicking?"*

### Create the Watershed Mosaic

The next cell is the first one that requests raster bytes. Depending on network speed, source resolution, output resolution, and watershed size, it may take several minutes.

The output filename records the acquisition year, HUC count, output resolution, and four-band RGBN content. Read the source-raster QA table after the transfer. All rows should report GeoTIFF-compatible metadata, four bands, unsigned-byte values, and source resolutions that agree with the inventory.

In [ ]:
resolution_label = str(OUTPUT_RESOLUTION_M).replace('.', 'p')
output_raster_path = Path(
    f"naip_{selected_year}_"
    f"{len(TARGET_HUCS)}huc12_"
    f"{resolution_label}m_rgbn.tif"
)

source_raster_qa, raster_coverage_pct = create_naip_subset(
    items=selected_items,
    aoi_projected=aoi_projected,
    output_path=output_raster_path,
    output_resolution_m=OUTPUT_RESOLUTION_M,
    minimum_coverage_pct=MINIMUM_COVERAGE_PCT,
)

print(f"\nSaved raster: {output_raster_path}")
print(
    f"Valid raster coverage of AOI cell centers: "
    f"{raster_coverage_pct:.2f}%"
)
source_raster_qa

The source QA table should show consistent four-band `uint8` rasters. Source EPSG and ground sample distance can differ from the final output CRS and 2 meter cell size because reprojection and resampling are part of the workflow.

The valid raster coverage should be at least the configured threshold. If footprint coverage passed but raster coverage failed, investigate unreadable pixels, source gaps, asset changes, or an inappropriate all-zero gap assumption.

## Part 8: Validate the Saved GeoTIFF

The final file is reopened from disk rather than trusted because the write call completed.

The checks below verify:

- GeoTIFF driver
- Four bands and `uint8` values
- Projected CRS
- Requested cell size
- Internal mask
- Valid pixel area compared with polygon area
- Band descriptions

Raster area uses the affine transform, not a hard-coded pixel area.

In [ ]:
with rasterio.open(output_raster_path) as source:
    saved_driver = source.driver
    saved_band_count = source.count
    saved_dtypes = source.dtypes
    saved_crs = source.crs
    saved_resolution = source.res
    saved_width = source.width
    saved_height = source.height
    saved_band_descriptions = source.descriptions
    saved_mask = source.dataset_mask() > 0
    saved_tags = source.tags()

    # Affine determinant gives cell area even if a grid contains rotation
    saved_cell_area_m2 = abs(
        source.transform.a * source.transform.e
        - source.transform.b * source.transform.d
    )

valid_raster_area_m2 = saved_mask.sum() * saved_cell_area_m2
area_difference_pct = (
    100
    * abs(valid_raster_area_m2 - aoi_area_m2)
    / aoi_area_m2
)
output_file_mb = output_raster_path.stat().st_size / 1024 ** 2

problems = []
if saved_driver != 'GTiff':
    problems.append(f"expected GTiff, found {saved_driver}")
if saved_band_count != 4:
    problems.append(f"expected 4 bands, found {saved_band_count}")
if any(dtype != 'uint8' for dtype in saved_dtypes):
    problems.append(f"expected uint8, found {saved_dtypes}")
if saved_crs != output_crs:
    problems.append(f"expected {output_crs}, found {saved_crs}")
if not np.allclose(
    np.abs(saved_resolution),
    (OUTPUT_RESOLUTION_M, OUTPUT_RESOLUTION_M),
    atol=0.001,
):
    problems.append(
        f"expected {OUTPUT_RESOLUTION_M} m cells, found "
        f"{saved_resolution}"
    )
if area_difference_pct > 1.0:
    problems.append(
        f"valid raster area differs from polygon area by "
        f"{area_difference_pct:.2f}%"
    )

print(f"Driver: {saved_driver}")
print(f"Dimensions: {saved_width:,} columns x {saved_height:,} rows")
print(f"Bands: {saved_band_count} {saved_band_descriptions}")
print(f"Data types: {saved_dtypes}")
print(f"CRS: {saved_crs}")
print(f"Cell size: {saved_resolution}")
print(f"Valid raster area: {valid_raster_area_m2 / 4046.8564224:,.0f} acres")
print(f"Polygon area: {aoi_area_acres:,.0f} acres")
print(f"Area difference: {area_difference_pct:.3f}%")
print(f"Compressed file size: {output_file_mb:,.1f} MB")

if problems:
    raise RuntimeError(
        "Saved-raster QA/QC failed:\n- " + "\n- ".join(problems)
    )

print("Saved-raster QA/QC passed.")

The raster and polygon areas will not be exactly equal because a raster represents the polygon with finite cell centers. At 2 meter resolution, the difference should be small. A difference above 1 percent stops the lesson.

The file size should be lower than the uncompressed estimate, but compression depends on image texture and should not be used as a validity check.

### Confirm Traceability Without Exposing Access

The next cell checks the metadata fields that should travel with the GeoTIFF. It also confirms that no token-like query signature was written to the tags.

The token expiration printed earlier is session information. It is not part of the imagery provenance.

In [ ]:
required_tag_names = {
    'source_program',
    'source_host',
    'stac_collection',
    'naip_year',
    'source_item_ids',
    'acquisition_dates',
    'access_date',
    'output_resolution_m',
    'processing',
    'source_reference',
}
missing_tags = sorted(required_tag_names - set(saved_tags))
if missing_tags:
    raise RuntimeError(f"Output metadata is missing tags: {missing_tags}")

tag_text = ' '.join(saved_tags.values()).lower()
token_markers = ['sig=', 'se=', 'sp=', 'sv=']
if any(marker in tag_text for marker in token_markers):
    raise RuntimeError(
        "A temporary access-token component was found in output metadata."
    )

metadata_review = pd.DataFrame({
    'field': sorted(required_tag_names),
    'value': [saved_tags[name] for name in sorted(required_tag_names)],
})
metadata_review

The metadata should identify the source program, host, collection, year, source Item IDs, acquisition dates, access date, output resolution, and processing method.

Signed URLs are intentionally absent. An Item ID is stable provenance. A short-lived access signature is not.

## Part 9: Visual Review

### Create Natural-Color and Color-Infrared Views

The saved GeoTIFF stores raw unsigned-byte image values. For display, the next cell reads a reduced preview and stretches each band between its 2nd and 98th valid percentiles.

The stretch affects only the PNG and notebook figure. It does not alter the GeoTIFF.

- Natural color uses red, green, and blue.
- Color infrared uses near infrared, red, and green. Healthy vegetation commonly appears red.

In [ ]:
def percentile_stretch(band_stack, lower=2, upper=98):
    '''Stretch a three-band masked array to display values from 0 to 1.'''
    display = np.zeros(band_stack.shape, dtype='float32')
    combined_mask = np.any(
        np.ma.getmaskarray(band_stack),
        axis=0,
    )

    for band_index in range(3):
        band = np.ma.array(
            band_stack[band_index],
            mask=combined_mask,
        )
        valid_values = band.compressed()
        if valid_values.size == 0:
            continue

        low_value, high_value = np.percentile(
            valid_values,
            [lower, upper],
        )
        if high_value <= low_value:
            high_value = low_value + 1

        display[band_index] = np.clip(
            (band.filled(low_value) - low_value)
            / (high_value - low_value),
            0,
            1,
        )

    display[:, combined_mask] = 1
    return np.moveaxis(display, 0, -1)


with rasterio.open(output_raster_path) as source:
    preview_scale = max(source.width, source.height) / 1400
    preview_width = max(1, round(source.width / preview_scale))
    preview_height = max(1, round(source.height / preview_scale))

    preview = source.read(
        [1, 2, 3, 4],
        out_shape=(4, preview_height, preview_width),
        resampling=Resampling.bilinear,
        masked=True,
    )
    image_extent = plotting_extent(source)

natural_color = percentile_stretch(preview[[0, 1, 2]])
color_infrared = percentile_stretch(preview[[3, 0, 1]])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for axis, image, title in [
    (axes[0], natural_color, 'Natural Color: Red, Green, Blue'),
    (axes[1], color_infrared, 'Color Infrared: NIR, Red, Green'),
]:
    axis.imshow(image, extent=image_extent)
    aoi_projected.boundary.plot(
        ax=axis,
        color='white',
        linewidth=1.2,
    )
    axis.set_title(
        f"NAIP {selected_year} {title}",
        fontweight='normal',
    )
    axis.set_xlabel('Easting, meters')
    axis.set_ylabel('Northing, meters')

plt.tight_layout()
preview_figure_path = Path(
    f"naip_{selected_year}_watershed_preview.png"
)
plt.savefig(preview_figure_path, dpi=180, bbox_inches='tight')
plt.show()

print(f"Saved preview: {preview_figure_path}")

### What to Inspect

Check the natural-color image for:

- Missing rectangular areas
- Strong seams that may indicate different acquisition dates or color balance
- Obvious shifts between roads, channels, and the watershed overlay
- Cloud, haze, shadows, or unusual seasonal conditions

Use the color-infrared view to distinguish vigorous vegetation from many built or bare surfaces. Do not treat red color as a calibrated vegetation measurement.

The preview is a visual QA/QC product. It is not a substitute for checking metadata, coverage, resolution, and field conditions.

## Part 10: Export the Source-Tile Inventory

The CSV records the stable information needed to reproduce and review the mosaic:

- Source Item ID
- Acquisition date
- NAIP year
- Ground sample distance
- Source EPSG
- State and band count
- Output filename and access date

The CSV does not contain raster URLs with temporary tokens.

In [ ]:
tile_inventory_path = Path(
    f"naip_{selected_year}_source_tile_inventory.csv"
)

tile_inventory_export = (
    selected_footprints[[
        'tile_number',
        'item_id',
        'acquisition_datetime',
        'naip_year',
        'ground_sample_distance_m',
        'source_epsg',
        'state',
        'band_count',
        'asset_type',
    ]]
    .copy()
)
tile_inventory_export['stac_collection'] = 'naip'
tile_inventory_export['source_program'] = (
    'USDA National Agriculture Imagery Program'
)
tile_inventory_export['source_host'] = (
    'Microsoft Planetary Computer'
)
tile_inventory_export['access_date'] = (
    datetime.date.today().isoformat()
)
tile_inventory_export['output_raster'] = (
    output_raster_path.name
)
tile_inventory_export['output_resolution_m'] = (
    OUTPUT_RESOLUTION_M
)

export_text = tile_inventory_export.to_csv(index=False).lower()
if any(marker in export_text for marker in token_markers):
    raise RuntimeError(
        "A temporary access-token component was found in the inventory."
    )

tile_inventory_export.to_csv(
    tile_inventory_path,
    index=False,
)

print(
    f"Saved tile inventory: {tile_inventory_path} "
    f"({len(tile_inventory_export)} rows)"
)
tile_inventory_export

The number of CSV rows should equal the selected tile count printed earlier. Confirm that every row has the selected NAIP year and a populated acquisition date.

The output GeoTIFF, tile inventory, and preview should be kept together during review.

### Download the Lesson Outputs

In Google Colab, the next cell starts a browser download for each file. In a local Jupyter environment, it prints the saved paths instead.

In [ ]:
output_files = [
    output_raster_path,
    tile_inventory_path,
    preview_figure_path,
]

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

for output_file in output_files:
    if not output_file.exists():
        raise FileNotFoundError(output_file)

    if colab_files is not None:
        colab_files.download(str(output_file))
    else:
        print(f"Local output ready: {output_file.resolve()}")

## Engineering Interpretation and Limitations

### What the Output Supports

The mosaic is suitable for watershed screening, visual review, map backgrounds, and identifying features that deserve closer investigation.

### What the Output Does Not Establish

- It does not prove drainage connectivity.
- It does not provide elevation or channel geometry.
- It does not directly measure impervious percentage.
- It does not replace current local imagery, survey, or field review.
- It does not establish conditions on a storm date outside the acquisition period.

### Acquisition and Processing Cautions

1. **Year is not one instant.** Tiles in one NAIP year can have different flight dates.
2. **Leaf-on imagery affects interpretation.** Vegetation can obscure channels, culverts, and ground condition.
3. **Resampling changes the teaching output, not the source.** The 2 meter GeoTIFF is coarser than the source Items.
4. **Tile seams can remain visible.** Different lighting, flight lines, or acquisition dates may produce color changes.
5. **Aerial imagery can have horizontal error.** Compare it with project-control data before detailed measurement.
6. **Large watersheds need a different processing strategy.** Use tiled outputs or a cloud workflow rather than one enormous in-memory mosaic.

## Troubleshooting

| Problem | Likely cause | What to inspect |
|---|---|---|
| No STAC Items are found | AOI is outside collection coverage or geometry is wrong | Print WGS84 bounds and verify longitude and latitude |
| Search stops during pagination | Temporary network or service issue | Rerun the search and inspect the status message |
| Newest year fails coverage | Only part of the AOI was flown or cataloged that year | Use the newest year passing the coverage threshold |
| Raster asset will not open | Token expired, network failed, or host is unavailable | Rerun the mosaic cell to request a fresh token |
| `You have exceeded a rate limit` (HTTP 429) | Too many anonymous requests from your address, which Colab shares between users | The helpers wait and retry automatically. If it persists, wait a minute and rerun the cell |
| Estimated array is too large | AOI is large or output resolution is too fine | Increase cell size, reduce AOI, or process tiles separately |
| Raster area differs from polygon area | Gaps, CRS problem, or mask issue | Check source footprint coverage, output CRS, and internal mask |
| Mosaic has strong seams | Different dates, lighting, or source balancing | Review the tile inventory and source footprints |
| NIR display looks wrong | Band order was changed | Confirm descriptions are Red, Green, Blue, Near infrared |
| Download cell does nothing locally | It is not running in Colab | Use the printed local paths |

## Practice Exercises

### Exercise 1: Select Another Watershed

Choose another HUC-12 value from the course file. Rerun the AOI, search, coverage, size, and mosaic cells. Record the selected year and source ground sample distance.

In [ ]:
# EXERCISE 1: Select another watershed
# Your code here.
#
# Steps:
# 1. Print available HUC12 and Name values.
# 2. Replace TARGET_HUCS with one different identifier.
# 3. Rerun every cell from area-of-interest selection onward.
# 4. Confirm footprint and raster coverage both pass.

### Exercise 2: Compare Output Resolutions

Run only the size-estimate calculation for 5 meter, 2 meter, 1 meter, and native source resolution. Create a small table with width, height, pixel count, and uncompressed megabytes.

Do not create every raster. The purpose is to make a processing decision before download.

In [ ]:
# EXERCISE 2: Compare output resolutions
# Your code here.
#
# Suggested resolutions:
# [5.0, 2.0, 1.0, source_gsd_min]
#
# For each resolution, calculate:
# width, height, total pixels, and four-band uncompressed MB.

### Exercise 3: Review an Earlier Complete Year

Choose an earlier year that passes the coverage check. Create a second mosaic and compare visible development, channel condition, or riparian cover.

Keep the comparison qualitative unless you develop a separate, validated image-classification method. Differences in season, lighting, resolution, and image processing can look like land-surface change.

In [ ]:
# EXERCISE 3: Review an earlier complete year
# Your code here.
#
# Steps:
# 1. Choose a year where passes_coverage_check is True.
# 2. Set REQUESTED_YEAR to that year.
# 3. Create a year-specific output filename.
# 4. Compare source dates, GSD, and visible conditions.

### Exercise 4: Use a Custom Project AOI

Replace the watershed selection with a public or approved project-area GeoJSON. Reproject it to EPSG:4326 for search and a suitable projected CRS for area and output.

Before retrieval, confirm:

- The geometry is valid.
- The source CRS is defined.
- The project has permission to use and share the boundary.
- The output-size estimate is acceptable.

In [ ]:
# EXERCISE 4: Use a custom AOI
# Your code here.
#
# Suggested starting pattern:
# custom_aoi = gpd.read_file('approved_project_aoi.geojson')
# print(custom_aoi.crs)
# print(custom_aoi.total_bounds)

### Challenge: Process Watersheds Separately

If `TARGET_HUCS` contains disconnected watersheds, one combined bounding rectangle can include a large amount of unused space. Build a loop that creates one output per HUC-12.

Suggested question for your AI assistant:

*"Help me refactor the tested NAIP workflow so each HUC-12 is searched, size-checked, downloaded, validated, and exported separately. Keep one acquisition year per watershed and do not log access tokens."*

Review the proposed changes carefully. Confirm that every output gets its own coverage checks and source inventory.

## Key Takeaways

- A STAC search returns metadata records before raster data.
- The newest individual tile is not necessarily the newest complete watershed acquisition.
- Coverage should be measured by year before a mosaic is built.
- Cloud Optimized GeoTIFFs allow spatially limited remote reads.
- Output size should be estimated before high-resolution imagery is transferred.
- Source ground sample distance and output cell size are different metadata.
- A completed download must still pass raster, mask, CRS, band, resolution, and area checks.
- Stable Item IDs belong in provenance records. Temporary access tokens do not.

## Next Steps

You are ready to:

- Apply the workflow to another approved watershed or project AOI.
- Compare two complete NAIP acquisitions with careful date and resolution review.
- Use imagery as visual evidence alongside Annual NLCD, soils, terrain, local GIS, and field observations.
- Replace the in-memory mosaic with a tiled workflow when project scale requires native-resolution imagery.